# Week 8 - Activity 1: Data and Pipeline Parallel Training

In this activity, we'll explore different parallel training strategies for transformers:
1. Data parallel training
2. Pipeline parallel training
3. Analyzing performance and memory usage

We'll use PyTorch's distributed training features to implement these approaches.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import numpy as np
from typing import Optional, List
import time

## 1. Simple Transformer Implementation

First, let's implement a basic transformer that we can parallelize:

In [ ]:
class SimpleTransformerLayer(nn.Module):
    def __init__(self, d_model: int = 512, nhead: int = 8, dim_feedforward: int = 2048):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.activation = nn.ReLU()
        
    def forward(self, x):
        # Self attention
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm1(x + attn_output)
        
        # Feed forward
        ff_output = self.linear2(self.activation(self.linear1(x)))
        x = self.norm2(x + ff_output)
        
        return x

class SimpleTransformer(nn.Module):
    def __init__(self, num_layers: int = 6, d_model: int = 512):
        super().__init__()
        self.layers = nn.ModuleList([
            SimpleTransformerLayer(d_model) for _ in range(num_layers)
        ])
        
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

## 2. Data Parallel Training

Let's implement data parallel training using DistributedDataParallel:

In [ ]:
def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("gloo", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

class DummyDataset(Dataset):
    def __init__(self, size=1000, seq_len=20, d_model=512):
        self.data = torch.randn(size, seq_len, d_model)
        self.labels = torch.randn(size, seq_len, d_model)
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

def train_data_parallel(rank, world_size):
    print(f"Running DDP on rank {rank}.")
    setup(rank, world_size)
    
    # Create model and move it to GPU with id rank
    model = SimpleTransformer()
    torch.cuda.set_device(rank)
    model.cuda(rank)
    model = DDP(model, device_ids=[rank])
    
    # Create data loader
    dataset = DummyDataset()
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, batch_size=32, sampler=sampler)
    
    # Training loop
    optimizer = torch.optim.Adam(model.parameters())
    criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(2):
        sampler.set_epoch(epoch)
        for batch_idx, (data, target) in enumerate(dataloader):
            data, target = data.cuda(rank), target.cuda(rank)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            if batch_idx % 10 == 0 and rank == 0:
                print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.6f}')
    
    cleanup()

def run_data_parallel(world_size):
    mp.spawn(train_data_parallel,
             args=(world_size,),
             nprocs=world_size,
             join=True)

## 3. Pipeline Parallel Training

Now let's implement pipeline parallelism by splitting the model across devices:

In [ ]:
class PipelineParallelTransformer(nn.Module):
    def __init__(self, num_layers: int = 6, d_model: int = 512, num_stages: int = 2):
        super().__init__()
        assert num_layers % num_stages == 0, "Number of layers must be divisible by number of stages"
        
        layers_per_stage = num_layers // num_stages
        self.stages = nn.ModuleList()
        
        for i in range(num_stages):
            stage_layers = nn.ModuleList([
                SimpleTransformerLayer(d_model) 
                for _ in range(layers_per_stage)
            ])
            self.stages.append(stage_layers)
    
    def forward(self, x, stage_id):
        # Process only the layers in the current stage
        for layer in self.stages[stage_id]:
            x = layer(x)
        return x

def train_pipeline_parallel(rank, world_size):
    print(f"Running pipeline parallel on rank {rank}")
    setup(rank, world_size)
    
    # Create model and move stage to appropriate device
    model = PipelineParallelTransformer(num_stages=world_size)
    stage = model.stages[rank].cuda(rank)
    
    # Create data
    batch_size = 32
    seq_len = 20
    d_model = 512
    
    # Training loop
    optimizer = torch.optim.Adam(stage.parameters())
    criterion = nn.MSELoss()
    
    for epoch in range(2):
        # Generate dummy batch
        if rank == 0:
            data = torch.randn(batch_size, seq_len, d_model).cuda(rank)
        else:
            data = torch.zeros(batch_size, seq_len, d_model).cuda(rank)
            
        # Pipeline forward pass
        optimizer.zero_grad()
        
        # Forward pass through current stage
        output = stage(data)
        
        # Send output to next stage
        if rank < world_size - 1:
            dist.send(output, rank + 1)
        if rank > 0:
            dist.recv(data, rank - 1)
        
        # Backward pass
        if rank == world_size - 1:
            target = torch.randn_like(output)
            loss = criterion(output, target)
            loss.backward()
            
            if epoch % 10 == 0:
                print(f'Epoch: {epoch}, Loss: {loss.item():.6f}')
        
        optimizer.step()
    
    cleanup()

def run_pipeline_parallel(world_size):
    mp.spawn(train_pipeline_parallel,
             args=(world_size,),
             nprocs=world_size,
             join=True)

## 4. Performance Analysis

Let's compare the performance of different parallelization strategies:

In [ ]:
def measure_memory_usage():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / 1024**2  # MB
    return 0

def run_performance_comparison():
    results = []
    world_sizes = [1, 2, 4]  # Number of GPUs to use
    
    for world_size in world_sizes:
        print(f"\nTesting with {world_size} GPUs:")
        
        # Data Parallel
        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()
        run_data_parallel(world_size)
        dp_time = time.time() - start_time
        dp_memory = measure_memory_usage()
        
        # Pipeline Parallel
        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()
        run_pipeline_parallel(world_size)
        pp_time = time.time() - start_time
        pp_memory = measure_memory_usage()
        
        results.append({
            'world_size': world_size,
            'dp_time': dp_time,
            'dp_memory': dp_memory,
            'pp_time': pp_time,
            'pp_memory': pp_memory
        })
    
    return pd.DataFrame(results)

# Run comparison if multiple GPUs are available
if torch.cuda.device_count() > 1:
    results_df = run_performance_comparison()
    print("\nPerformance Results:")
    print(results_df)
    
    # Plot results
    import matplotlib.pyplot as plt
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Training time comparison
    ax1.plot(results_df['world_size'], results_df['dp_time'], marker='o', label='Data Parallel')
    ax1.plot(results_df['world_size'], results_df['pp_time'], marker='s', label='Pipeline Parallel')
    ax1.set_xlabel('Number of GPUs')
    ax1.set_ylabel('Training Time (s)')
    ax1.set_title('Training Time vs. Number of GPUs')
    ax1.legend()
    ax1.grid(True)
    
    # Memory usage comparison
    ax2.plot(results_df['world_size'], results_df['dp_memory'], marker='o', label='Data Parallel')
    ax2.plot(results_df['world_size'], results_df['pp_memory'], marker='s', label='Pipeline Parallel')
    ax2.set_xlabel('Number of GPUs')
    ax2.set_ylabel('Peak Memory Usage per GPU (MB)')
    ax2.set_title('Memory Usage vs. Number of GPUs')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("This notebook requires multiple GPUs for parallel training demonstration.")

## Discussion Points

1. Parallelization Strategy Selection
   - When to use data parallel vs. pipeline parallel?
   - What are the communication patterns in each approach?
   - How do they scale with model size vs. batch size?

2. Performance Analysis
   - What's the impact on training time?
   - How does memory usage differ?
   - What are the bottlenecks in each approach?

3. Implementation Considerations
   - How to handle uneven pipeline stages?
   - What's the impact of micro-batch size?
   - How to optimize communication?

4. Practical Applications
   - Scaling to very large models
   - Combining multiple parallelism strategies
   - Hardware considerations